# 04 — Netherlands: Motorway AnalysisAnalyse RWS accident data. `MAXSNELHD` (speed limit) directly identifies motorway accidents — no spatial join needed.

In [ ]:
from pathlib import Pathimport pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsimport syssys.path.insert(0, '../src')from autobahn_safety.analysis import severity_indexDATA_RAW = Path('../data/raw')DATA_PROC = Path('../data/processed')DATA_PROC.mkdir(parents=True, exist_ok=True)sns.set_theme(style='whitegrid')print('Ready.')

## 1. Load RWS accident data

In [ ]:
rws_files = sorted((DATA_RAW / 'netherlands' / 'rws').glob('rws_accidents_*.csv'))print(f'Available year files: {[f.stem.split("_")[-1] for f in rws_files]}')df_rws = pd.concat([pd.read_csv(f) for f in rws_files], ignore_index=True)df_rws['year'] = df_rws['JAAR_VKL']print(f'Total: {len(df_rws):,} records, years {df_rws["year"].min()}–{df_rws["year"].max()}')df_rws.head(3)

## 2. Classify road types by speed limitDutch motorways (`autosnelwegen`) have a posted speed limit of 100 or 130 km/h.- 130 km/h → standard motorway (daytime)- 100 km/h → motorway with lower limit (night / environmental zone)- <100 km/h outside built-up area → N-road (`nationale weg`)- 50/30 km/h → urban

In [ ]:
def classify_road_nl(row):    spd = row['MAXSNELHD']    if pd.isna(spd):        return 'unknown'    spd = int(spd)    if spd in (100, 120, 130):        return 'motorway'    elif spd in (60, 70, 80, 90) and row.get('BEBKOM') == 'Buiten':        return 'n_road'    elif spd == 50:        return 'urban_50'    elif spd == 30:        return 'urban_30'    else:        return 'other'df_rws['road_type'] = df_rws.apply(classify_road_nl, axis=1)print(df_rws['road_type'].value_counts().to_string())

In [ ]:
# Motorway subsetdf_mw_nl = df_rws[df_rws['road_type'] == 'motorway'].copy()print(f'Motorway accidents: {len(df_mw_nl):,}')print(f'Speed limit breakdown:\n{df_mw_nl["MAXSNELHD"].value_counts().to_string()}')

## 3. Annual trends on NL motorways

In [ ]:
annual_nl = df_mw_nl.groupby('year').agg(    total=('AP3_CODE', 'count'),    fatal=('AP3_CODE', lambda x: (x == 'Dodelijk').sum()),    injury=('AP3_CODE', lambda x: (x == 'Letsel').sum()),    material=('AP3_CODE', lambda x: (x == 'Uitsluitend materiele schade').sum()),).reset_index()annual_nl['severity_idx'] = annual_nl['fatal'] / annual_nl['total']annual_nl

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))axes[0].plot(annual_nl['year'], annual_nl['total'], marker='o', color='#2a9d8f', linewidth=2, label='Total')axes[0].plot(annual_nl['year'], annual_nl['injury'], marker='s', color='#e9c46a', linewidth=2, label='Injury')axes[0].plot(annual_nl['year'], annual_nl['fatal'], marker='^', color='crimson', linewidth=2, label='Fatal')axes[0].set_title('NL motorway accidents by severity')axes[0].set_xlabel('Year'); axes[0].set_ylabel('Accidents')axes[0].legend()axes[1].plot(annual_nl['year'], annual_nl['severity_idx'] * 1000, marker='o', color='crimson', linewidth=2)axes[1].set_title('NL motorway: severity index (fatalities per 1000 accidents)')axes[1].set_xlabel('Year'); axes[1].set_ylabel('Fatalities per 1000 accidents')plt.suptitle('Netherlands motorway safety (RWS BRON)', fontweight='bold')plt.tight_layout(); plt.show()

## 4. Save processed NL data

In [ ]:
out = DATA_PROC / 'rws_motorway_annual.parquet'annual_nl.to_parquet(out, index=False)print(f'Saved to {out}')annual_nl